# NBA Game Outcome Predictor — Analyse exploratoire

Ce notebook explore les données produites par le pipeline avant toute modélisation :

1. Charger le brut *équipe × match* et le dataset de *features* niveau match.
2. Quantifier l'**avantage du terrain** (global et par saison).
3. Regarder la distribution du **defensive rating** estimé.
4. Mesurer la **corrélation des features avec la cible** (`home_win`).
5. Vérifier l'effet **repos / back-to-back**.

> Prérequis : avoir lancé `python src/data_ingestion.py` puis `python src/features.py`.
> Les données ne sont pas versionnées (`.gitignore` → `data/`).

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = pd.read_parquet(PROJECT_ROOT / "data" / "raw" / "games_team_level.parquet")
GAMES = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "features.parquet")

print(f"Brut equipe x match : {RAW.shape[0]} lignes ({RAW['GAME_ID'].nunique()} matchs)")
print(f"Features niveau match : {GAMES.shape[0]} lignes")
GAMES.head()

## 1. Avantage du terrain

La cible `home_win` vaut 1 si l'équipe à domicile gagne. C'est aussi la prédiction
de la **baseline** « domicile gagne toujours » : son accuracy = le taux de victoires
à domicile. C'est la barre que le modèle doit battre.

In [ ]:
overall = GAMES['home_win'].mean()
print(f"Taux de victoire a domicile (global) : {overall:.3f}")

by_season = GAMES.groupby('SEASON')['home_win'].mean()
ax = by_season.plot(kind='bar', figsize=(9, 4), color='#1f77b4')
ax.axhline(0.5, ls='--', color='grey', label='50 % (pile ou face)')
ax.set_ylabel('Taux de victoire domicile')
ax.set_title('Avantage du terrain par saison')
ax.legend()
plt.tight_layout(); plt.show()

On observe un avantage du terrain net (~55-60 %), **sauf 2020-21** : saison COVID
jouée en grande partie sans public (la « bulle » / arènes vides) — l'avantage du
terrain s'effondre, ce qui est un signal physique réel et rassurant sur les données.

## 2. Distribution du defensive rating

Rappel : `DEF_RATING = 100 x points encaisses / possessions estimees`. Plus c'est
**bas**, meilleure est la défense. On le calcule par match (côté brut).

In [ ]:
ax = RAW['DEF_RATING'].plot(kind='hist', bins=50, figsize=(8, 4), color='#ff7f0e', alpha=0.8)
ax.axvline(RAW['DEF_RATING'].median(), color='black', ls='--', label=f"mediane = {RAW['DEF_RATING'].median():.1f}")
ax.set_xlabel('Defensive rating (par match)')
ax.set_title('Distribution du defensive rating estime')
ax.legend()
plt.tight_layout(); plt.show()
RAW['DEF_RATING'].describe()

## 3. Corrélation des features avec la cible

Toutes les features sont des **différences domicile − extérieur**, décalées dans le
temps (aucune info du match courant). On regarde leur corrélation avec `home_win`.

In [ ]:
feature_cols = ['rest_days_diff', 'back_to_back_diff', 'form_5_diff',
                'def_rating_diff_5', 'off_rating_diff_5']
corr = GAMES[feature_cols + ['home_win']].corr()['home_win'].drop('home_win').sort_values()
ax = corr.plot(kind='barh', figsize=(7, 3.5), color='#2ca02c')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Correlation des features avec home_win')
plt.tight_layout(); plt.show()
corr

`off_rating_diff_5` corrèle positivement (mieux attaquer récemment → gagner) et
`def_rating_diff_5` **négativement** (un rating défensif plus bas = meilleure défense
→ plus de victoires). Cohérent avec les coefficients appris par la régression logistique.

## 4. Effet du repos

Le domicile gagne-t-il plus souvent quand il est **plus reposé** que l'adversaire ?

In [ ]:
bins = pd.cut(GAMES['rest_days_diff'], [-11, -1, 0, 1, 11],
              labels=['domicile - repose', 'egal (-1..0)', 'egal (0..1)', 'domicile + repose'])
win_by_rest = GAMES.groupby(bins, observed=True)['home_win'].mean()
ax = win_by_rest.plot(kind='bar', figsize=(8, 4), color='#9467bd')
ax.axhline(GAMES['home_win'].mean(), ls='--', color='grey', label='moyenne globale')
ax.set_ylabel('Taux de victoire domicile')
ax.set_title('Victoire a domicile selon le differentiel de repos')
ax.legend()
plt.tight_layout(); plt.show()

## Conclusion

- L'avantage du terrain est réel (~57 %) et **dépend du public** (effondrement 2020-21).
- Les ratings offensif/défensif récents sont les signaux les plus forts.
- Le repos joue à la marge, dans le sens attendu.

→ Ces features alimentent la régression logistique de `src/train.py`, évaluée par
split **temporel** dans `src/evaluate.py`. Le modèle bat la baseline « domicile gagne
toujours » de plusieurs points d'accuracy.